# Demo 3 — Verify a structural melt/pivot round trip

**Learning objectives**

- State the row grain and identifier/measured variables in wide and long forms.
- Use `melt()` to move repeated-measure columns into rows without aggregation.
- Use structural `pivot()` only after verifying identifier-variable uniqueness, then compare the reconstruction exactly with the source.

Colab is the default launch experience; local Jupyter runs the same cells. See `DEMO_GUIDE.md` for launch and rehearsal instructions. GitHub source opened in Colab is not automatically updated by edits in the Colab tab.

Compatibility candidate: Python 3.12.13, NumPy 2.0.2, pandas 3.0.3. This is not the final course lock until fresh local and Colab certification is complete. The fixture contains invented teaching records only.

In [ ]:
from importlib.metadata import PackageNotFoundError, version
import subprocess
import sys

PANDAS_CANDIDATE = "3.0.3"
try:
    installed_pandas = version("pandas")
except PackageNotFoundError:
    installed_pandas = None
if installed_pandas != PANDAS_CANDIDATE:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", f"pandas=={PANDAS_CANDIDATE}"],
        check=True,
    )

import numpy as np
import pandas as pd
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)

## Resolve and verify one wide table

The wide input has grain one row per participant and site. `participant_id` and `site_code` are identifier variables. `baseline_score` and `followup_score` are measured variables stored in separate columns.

In [ ]:
from hashlib import sha256
from pathlib import Path

SOURCE_RELATIVE_PATH = Path("06") / "demo" / "data" / "scores_wide.csv"
EXPECTED_SHA256 = "5098b5e9a0165f9f7f6e22bc761f01d5ddb7205af9b88d41876f116cea2d7c38"
SUPPLIED_SOURCE_BYTES = (
    b"participant_id,site_code,baseline_score,followup_score\n"
    b"P01,N,10.0,12.0\n"
    b"P02,S,8.5,9.5\n"
    b"P03,W,11.0,13.0\n"
)


def find_course_file(start, relative_path):
    current = start.resolve()
    while True:
        candidate = current / relative_path
        if candidate.is_file():
            return candidate
        if current.parent == current:
            return None
        current = current.parent


DATA_PATH = find_course_file(Path.cwd(), SOURCE_RELATIVE_PATH)
lecture_readme = find_course_file(Path.cwd(), Path("06") / "README.md")
if DATA_PATH is None:
    data_dir = Path.cwd() / "data"
    data_dir.mkdir(parents=True, exist_ok=True)
    DATA_PATH = data_dir / "scores_wide.csv"
    DATA_PATH.write_bytes(SUPPLIED_SOURCE_BYTES)
assert sha256(DATA_PATH.read_bytes()).hexdigest() == EXPECTED_SHA256

demo_base = Path.cwd() if lecture_readme is None else lecture_readme.parent / "demo"
OUTPUT_DIR = demo_base / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
wide_scores = pd.read_csv(
    DATA_PATH,
    dtype={"participant_id": "string", "site_code": "string"},
)
assert not wide_scores.duplicated(subset=["participant_id", "site_code"]).any()
print("Input:", DATA_PATH)
wide_scores

## Melt repeated measurements into rows

The long result has grain one row per participant, site, and measurement label. `melt()` keeps the identifier variables and moves the two measured column names into `visit_label`, with their values in `score`. It does not summarize or average anything.

In [ ]:
long_scores = wide_scores.melt(
    id_vars=["participant_id", "site_code"],
    value_vars=["baseline_score", "followup_score"],
    var_name="visit_label",
    value_name="score",
)
long_scores["visit_label"] = long_scores["visit_label"].astype("string")
assert len(long_scores) == len(wide_scores) * 2 == 6
assert not long_scores.duplicated(subset=["participant_id", "site_code", "visit_label"]).any()
long_scores

## Pivot only unique combinations back to columns

Structural `pivot()` requires at most one score for each (`participant_id`, `site_code`, `visit_label`) combination. When that key is unique, the operation can reconstruct the original values without aggregation.

In [ ]:
round_trip_scores = (
    long_scores
    .pivot(
        index=["participant_id", "site_code"],
        columns="visit_label",
        values="score",
    )
    .reset_index()
)
round_trip_scores.columns.name = None
round_trip_scores = round_trip_scores.loc[:, wide_scores.columns]
round_trip_scores.columns = wide_scores.columns.tolist()
expected_scores = wide_scores.sort_values(["participant_id", "site_code"]).reset_index(drop=True)
round_trip_scores = round_trip_scores.sort_values(["participant_id", "site_code"]).reset_index(drop=True)
pd.testing.assert_frame_equal(round_trip_scores, expected_scores)
round_trip_scores

## Make the uniqueness failure observable

A planted repeated identifier-variable combination makes `pivot()` refuse to guess. Lecture 08 later introduces aggregating `pivot_table()` for questions where combining repeated values is justified.

In [ ]:
duplicate_long_scores = pd.concat([long_scores, long_scores.iloc[[0]]], ignore_index=True)
duplicate_pivot_failed = False
try:
    duplicate_long_scores.pivot(
        index=["participant_id", "site_code"],
        columns="visit_label",
        values="score",
    )
except ValueError as error:
    duplicate_pivot_failed = True
    print("Expected pivot failure:", type(error).__name__)
assert duplicate_pivot_failed

## Replace and verify the long-form artifact

Write the validated long table without a positional index, read it back with the declared schema, and compare it exactly with the in-memory result.

In [ ]:
LONG_PATH = OUTPUT_DIR / "scores_long.csv"
long_scores.to_csv(LONG_PATH, index=False)
long_round_trip = pd.read_csv(
    LONG_PATH,
    dtype={"participant_id": "string", "site_code": "string", "visit_label": "string", "score": "float64"},
)
pd.testing.assert_frame_equal(long_round_trip, long_scores)
print("Demo 3 structural reshape round trip passed")
print(LONG_PATH)